In [2]:
import polars as pl
import numpy as np
from sklearn.linear_model import BayesianRidge
import pandas as pd

from sklearn.linear_model import Ridge
import os
import warnings
warnings.filterwarnings('ignore')

import random
def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)
seed_everything(seed=2024)

In [3]:
def custom_metric(y_true,y_pred,weight):
    weighted_r2=1-(np.sum(weight*(y_true-y_pred)**2)/np.sum(weight*y_true**2))
    return weighted_r2

train=pl.scan_parquet("./kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/**/*.parquet").collect().to_pandas()

cols=[f'feature_0{i}' if i<10 else f'feature_{i}' for i in range(79)]
X=train[cols].fillna(3).values
y=train['responder_6'].values
weights = train['weight'].values
print("train test split")

# Use only the last 10 million rows
split1 = 15000000
# Use the last 2 million rows of the 10 million for validation
split2 = 3000000

train_X = X[-split1:-split2]
train_y = y[-split1:-split2]
train_weight = weights[-split1:-split2]

test_X = X[-split2:]
test_y = y[-split2:]
test_weight = weights[-split2:]

print(f"train_X.shape: {train_X.shape}, test_X.shape: {test_X.shape}")
print("fit and predict")

train test split
train_X.shape: (12000000, 79), test_X.shape: (3000000, 79)
fit and predict


In [4]:
import optuna
del train

In [5]:
def objective(trial):
    params = {
        "alpha_1": trial.suggest_float("alpha_1", 1e-10, 10, log=True),
        "alpha_2": trial.suggest_float("alpha_2", 1e-10, 10, log=True),
        "lambda_1": trial.suggest_float("lambda_1", 1e-10, 10, log=True),
        "lambda_2": trial.suggest_float("lambda_2", 1e-10, 10, log=True),
        "max_iter": trial.suggest_int("max_iter", 100, 1000),
        "tol": trial.suggest_float("tol", 1e-5, 1, log=True),
        # "alpha_init": trial.suggest_uniform("alpha_init", 1e-4, 5),
        # "lambda_init": trial.suggest_uniform("lambda_init", 1e-4, 5),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
    }

    model = BayesianRidge(**params)
    model.fit(train_X,train_y, sample_weight=train_weight)
    y_pred = model.predict(test_X)
    score = custom_metric(test_y,y_pred, test_weight)
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=500)

print("Best trial:")
print("  Value: ", study.best_trial.value)
print("  Params: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

[I 2024-11-15 11:13:56,758] A new study created in memory with name: no-name-108ced06-36a8-418c-8bdc-bcbc02b4bd87
[I 2024-11-15 11:14:10,814] Trial 0 finished with value: 0.003214597702026367 and parameters: {'alpha_1': 0.4981766213119189, 'alpha_2': 0.525588252152103, 'lambda_1': 2.831977912651812e-06, 'lambda_2': 1.7165300534494608e-05, 'max_iter': 521, 'tol': 0.011182600613003284, 'fit_intercept': True}. Best is trial 0 with value: 0.003214597702026367.
[I 2024-11-15 11:14:24,200] Trial 1 finished with value: 0.003214597702026367 and parameters: {'alpha_1': 3.8160804113243345, 'alpha_2': 0.0007179325204471017, 'lambda_1': 6.379522292944697e-06, 'lambda_2': 5.357791494078835e-08, 'max_iter': 466, 'tol': 0.012682818754894565, 'fit_intercept': True}. Best is trial 0 with value: 0.003214597702026367.
[I 2024-11-15 11:14:37,281] Trial 2 finished with value: 0.003184199333190918 and parameters: {'alpha_1': 1.2054299514389587e-07, 'alpha_2': 3.5482199117419375, 'lambda_1': 3.23080152276097

Best trial:
  Value:  0.003215312957763672
  Params: 
    alpha_1: 1.3243409416326203e-06
    alpha_2: 2.162069996737851e-08
    lambda_1: 9.39211663214071
    lambda_2: 3.538409339292369e-08
    max_iter: 816
    tol: 0.0025204904466949373
    fit_intercept: True


In [6]:
params = study.best_params

model = BayesianRidge(**params)
model.fit(train_X,train_y, sample_weight=train_weight)
y_pred = model.predict(test_X)
score = custom_metric(test_y,y_pred, test_weight)
print(f"Score: {score}")

Score: 0.003215312957763672


In [7]:
params

{'alpha_1': 1.3243409416326203e-06,
 'alpha_2': 2.162069996737851e-08,
 'lambda_1': 9.39211663214071,
 'lambda_2': 3.538409339292369e-08,
 'max_iter': 816,
 'tol': 0.0025204904466949373,
 'fit_intercept': True}

In [8]:
# save model
import joblib
joblib.dump(model, "model_2.pkl")

['model_2.pkl']